# FCC BDC Processing — June 2022 Filing

Processes the **June 2022** FCC Broadband Data Collection (BDC) filing for the 5 project states (CA, GA, IL, NY, TX). This filing contains **Fiber to the Premises (FTTP)** data only.

**Pipeline:** Load raw location-level CSVs → Map providers via master list → Roll up to census block level → Export.

**Input Files:**
- `data/FCC Previous Years/June 2022/bdc_XX_FibertothePremises_fixed_broadband_J22_10may2024.csv` (5 states)
- `output/final_master_set.csv` (provider → holding company mapping)

**Output:** `output/all_states_block_level_availability_202206.csv`

**Roll-up logic:**
```
For each unique (block_geoid, provider_id, holding_company, technology):
    → MAX of max_advertised_download_speed
    → MAX of max_advertised_upload_speed
    → Flag whether residential, business, or both are served
    → MAX of low_latency
```

## 1. Setup & Imports

In [1]:
import os
import time
import pandas as pd

pd.set_option('display.max_columns', 20)

# --- Paths ---
DATA_DIR = '../../data/FCC Previous Years/June 2022'
MASTER_FILE = '../../output/final_master_set.csv'
OUTPUT_DIR = '../../output'

FILING_PERIOD = '2022-06'

STATE_FILES = {
    'CA': os.path.join(DATA_DIR, 'bdc_06_FibertothePremises_fixed_broadband_J22_10may2024.csv'),
    'GA': os.path.join(DATA_DIR, 'bdc_13_FibertothePremises_fixed_broadband_J22_10may2024.csv'),
    'IL': os.path.join(DATA_DIR, 'bdc_17_FibertothePremises_fixed_broadband_J22_10may2024.csv'),
    'NY': os.path.join(DATA_DIR, 'bdc_36_FibertothePremises_fixed_broadband_J22_10may2024.csv'),
    'TX': os.path.join(DATA_DIR, 'bdc_48_FibertothePremises_fixed_broadband_J22_10may2024.csv'),
}

TECHNOLOGY_MAP = {
    10: 'Copper Wire (DSL)',
    40: 'Cable',
    50: 'Fiber to the Premises (FTTP)'
}

# Verify files exist
for state, fp in STATE_FILES.items():
    exists = os.path.exists(fp)
    print(f"  {state}: {'OK' if exists else 'MISSING'} — {os.path.basename(fp)}")

  CA: OK — bdc_06_FibertothePremises_fixed_broadband_J22_10may2024.csv
  GA: OK — bdc_13_FibertothePremises_fixed_broadband_J22_10may2024.csv
  IL: OK — bdc_17_FibertothePremises_fixed_broadband_J22_10may2024.csv
  NY: OK — bdc_36_FibertothePremises_fixed_broadband_J22_10may2024.csv
  TX: OK — bdc_48_FibertothePremises_fixed_broadband_J22_10may2024.csv


## 2. Load All State CSVs

In [2]:
frames = []

for state, filepath in STATE_FILES.items():
    t0 = time.time()
    print(f"Loading {state}...", end=' ', flush=True)
    df = pd.read_csv(filepath)
    elapsed = time.time() - t0
    print(f"{df.shape[0]:,} rows in {elapsed:.1f}s")
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
del frames

print(f"\nTotal raw records: {raw.shape[0]:,}")
print(f"Columns: {list(raw.columns)}")
raw.head()

Loading CA... 3,695,477 rows in 6.6s
Loading GA... 1,625,832 rows in 3.4s
Loading IL... 1,311,555 rows in 2.8s
Loading NY... 3,180,509 rows in 5.6s
Loading TX... 4,949,267 rows in 9.1s

Total raw records: 14,762,640
Columns: ['frn', 'provider_id', 'brand_name', 'location_id', 'technology', 'max_advertised_download_speed', 'max_advertised_upload_speed', 'low_latency', 'business_residential_code', 'state_usps', 'block_geoid', 'h3_res8_id']


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,1545748,131057,The Ponderosa Internet,1083255296,50,100,50,1,X,CA,60390001021132,8829ab0901fffff
1,1549914,131441,Volcano Vision,1286496665,50,100,100,1,R,CA,60050003042033,88283665c7fffff
2,1549914,131441,Volcano Vision,1286505959,50,100,100,1,R,CA,60050001012006,882832dab5fffff
3,17793639,440162,Omsoft,1315296891,50,100,100,1,X,CA,61130105052025,882832b6c3fffff
4,14817357,350042,UPN,1344669706,50,1000,1000,1,B,CA,60816092022003,8828342939fffff


## 3. Basic Data Checks

In [3]:
print("--- Data Types ---")
print(raw.dtypes)

print("\n--- Missing Values ---")
print(raw.isnull().sum())

print(f"\n--- Technology Codes ---")
print(raw['technology'].value_counts())

print(f"\n--- Unique Providers ---")
print(f"provider_id: {raw['provider_id'].nunique()}")
print(f"brand_name:  {raw['brand_name'].nunique()}")

print(f"\n--- States ---")
print(raw['state_usps'].value_counts())

print(f"\n--- Business/Residential Split ---")
print(raw['business_residential_code'].value_counts())

--- Data Types ---
frn                              int64
provider_id                      int64
brand_name                         str
location_id                      int64
technology                       int64
max_advertised_download_speed    int64
max_advertised_upload_speed      int64
low_latency                      int64
business_residential_code          str
state_usps                         str
block_geoid                      int64
h3_res8_id                         str
dtype: object

--- Missing Values ---
frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_speed      0
low_latency                      0
business_residential_code        0
state_usps                       0
block_geoid                      0
h3_res8_id                       0
dtype: int64

--- Technology Codes ---
technology
50   

## 4. Provider Mapping — Merge with Master Provider List

In [4]:
### Load provider master and deduplicate by provider_id
providers_master = pd.read_csv(MASTER_FILE)
providers_master.rename(columns={'Provider ID': 'provider_id', 'FRN': 'frn'}, inplace=True)
print(f"Master provider records (raw): {providers_master.shape[0]:,}")

### Deduplicate: keep first holding_company per provider_id
providers_dedup = providers_master.drop_duplicates(subset='provider_id', keep='first')[['provider_id', 'holding_company']]
print(f"Unique provider_id mappings: {providers_dedup.shape[0]:,}")

### Merge on provider_id only (FRN may differ across filings)
raw = raw.merge(providers_dedup, on='provider_id', how='left')

### Check unmatched providers
unmatched_mask = raw['holding_company'].isna()
unmatched_count = unmatched_mask.sum()
print(f"\nUnmatched rows: {unmatched_count:,} ({unmatched_count / len(raw) * 100:.2f}%)")

if unmatched_count > 0:
    unmatched_providers = raw.loc[unmatched_mask, ['provider_id', 'brand_name']].drop_duplicates()
    print(f"Unmatched provider_ids: {unmatched_providers.shape[0]}")
    print(unmatched_providers.to_string(index=False))
    
    ### Fallback: use brand_name as holding_company for unmatched
    raw.loc[unmatched_mask, 'holding_company'] = raw.loc[unmatched_mask, 'brand_name'].str.strip()
    print(f"\nFilled {unmatched_count:,} rows with brand_name as fallback.")

print(f"\nNull holding_company remaining: {raw['holding_company'].isna().sum()}")

Master provider records (raw): 2,883
Unique provider_id mappings: 2,179

Unmatched rows: 506,582 (3.43%)
Unmatched provider_ids: 47
 provider_id                               brand_name
      400092                   Crown Castle Fiber LLC
      450139                            WillitsOnline
      430059                                   Netrix
      130278               San Bruno CityNet Services
      230042                     S-Net Communications
      310090                               Telesystem
      450135                             Two Rock LAN
      390163                                NGN Fiber
      131413                 Georgia Windstream, LLC.
      131413   Windstream Georgia Communications, LLC
      300115                      Parker FiberNet LLC
      131413        Windstream Georgia Telephone, LLC
      131413                 Windstream Standard, LLC
      131413                  Windstream Georgia, LLC
      131413         Windstream Accucomm Telecom, LLC
    

## 5. Data Cleaning — block_geoid Preparation

In [5]:
### Check for null block_geoid
null_geoid = raw['block_geoid'].isna().sum()
print(f"Null block_geoid rows: {null_geoid:,}")

if null_geoid > 0:
    raw = raw.dropna(subset=['block_geoid'])
    print(f"Dropped {null_geoid:,} rows. Remaining: {raw.shape[0]:,}")

### Zero-pad block_geoid to 15 digits
raw['block_geoid'] = raw['block_geoid'].astype(int).astype(str).str.zfill(15)

### Validate length
bad_len = (raw['block_geoid'].str.len() != 15).sum()
print(f"Invalid block_geoid (not 15 digits): {bad_len:,}")

### Derive FIPS hierarchy
raw['state_fips'] = raw['block_geoid'].str[:2]
raw['county_fips'] = raw['block_geoid'].str[:5]
raw['tract_fips'] = raw['block_geoid'].str[:11]

print(f"\nState FIPS values: {sorted(raw['state_fips'].unique())}")
print(f"Unique census blocks: {raw['block_geoid'].nunique():,}")

Null block_geoid rows: 0
Invalid block_geoid (not 15 digits): 0

State FIPS values: ['06', '13', '17', '36', '48']
Unique census blocks: 720,672


## 6. Census Block Level Roll-Up

In [6]:
t0 = time.time()

block_level = raw.groupby(
    ['block_geoid', 'provider_id', 'holding_company', 'technology']
).agg(
    max_download=('max_advertised_download_speed', 'max'),
    max_upload=('max_advertised_upload_speed', 'max'),
    serves_residential=('business_residential_code', lambda x: any(v in ['R', 'X'] for v in x)),
    serves_business=('business_residential_code', lambda x: any(v in ['B', 'X'] for v in x)),
    low_latency=('low_latency', 'max'),
    state_fips=('state_fips', 'first'),
    county_fips=('county_fips', 'first'),
    tract_fips=('tract_fips', 'first')
).reset_index()

elapsed = time.time() - t0
print(f"Roll-up completed in {elapsed:.1f}s")
print(f"Location-level rows: {raw.shape[0]:,}")
print(f"Block-level rows:    {block_level.shape[0]:,}")
print(f"Reduction:           {raw.shape[0] / block_level.shape[0]:.1f}x")

### Free raw data memory
del raw

Roll-up completed in 125.7s
Location-level rows: 14,762,640
Block-level rows:    840,512
Reduction:           17.6x


## 7. Map Technology Codes & Add Filing Period

In [7]:
### Map technology codes
block_level['technology_name'] = block_level['technology'].map(TECHNOLOGY_MAP)

unmapped = block_level['technology_name'].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows with unmapped technology codes")
    print(block_level.loc[block_level['technology_name'].isna(), 'technology'].unique())
else:
    print(f"All technology codes mapped. Codes: {sorted(block_level['technology'].unique())}")

### Add filing period
block_level['filing_period'] = FILING_PERIOD

print(f"\n--- Technology Distribution ---")
print(block_level['technology_name'].value_counts())

All technology codes mapped. Codes: [np.int64(50)]

--- Technology Distribution ---
technology_name
Fiber to the Premises (FTTP)    840512
Name: count, dtype: int64


## 8. Summary Statistics

In [8]:
STATE_NAMES = {'06': 'CA', '13': 'GA', '17': 'IL', '36': 'NY', '48': 'TX'}

### Per-state block counts
print("=== Per-State Summary ===")
state_summary = block_level.groupby('state_fips').agg(
    rows=('block_geoid', 'size'),
    unique_blocks=('block_geoid', 'nunique'),
    unique_providers=('provider_id', 'nunique'),
    unique_companies=('holding_company', 'nunique')
).reset_index()
state_summary['state'] = state_summary['state_fips'].map(STATE_NAMES)
print(state_summary[['state', 'rows', 'unique_blocks', 'unique_providers', 'unique_companies']].to_string(index=False))

print(f"\n--- Overall ---")
print(f"Total block-level rows: {block_level.shape[0]:,}")
print(f"Unique census blocks:   {block_level['block_geoid'].nunique():,}")
print(f"Unique providers:       {block_level['provider_id'].nunique()}")
print(f"Unique companies:       {block_level['holding_company'].nunique()}")

### Top 15 providers
print(f"\n=== Top 15 Providers ===")
top = block_level['holding_company'].value_counts().head(15).reset_index()
top.columns = ['holding_company', 'rows']
top['pct'] = (top['rows'] / block_level.shape[0] * 100).round(1)
print(top.to_string(index=False))

=== Per-State Summary ===
state   rows  unique_blocks  unique_providers  unique_companies
   CA 180370         162343                63                63
   GA  89764          80126                74                76
   IL 105386          94130                82                82
   NY 191913         144588                53                53
   TX 273079         239485               128               132

--- Overall ---
Total block-level rows: 840,512
Unique census blocks:   720,672
Unique providers:       319
Unique companies:       327

=== Top 15 Providers ===
                       holding_company   rows  pct
                             AT&T Inc. 259774 30.9
           Verizon Communications Inc.  88349 10.5
   Frontier Communications Corporation  87702 10.4
                Charter Communications  40685  4.8
                            Altice USA  36685  4.4
                  Radiate Holdings, LP  17518  2.1
                Stratus Networks, Inc.  15634  1.9
                   

## 9. Validation

In [9]:
print("=== VALIDATION REPORT ===")
print()

# 1. Null block_geoid
null_geoid = block_level['block_geoid'].isna().sum()
print(f"[{'PASS' if null_geoid == 0 else 'FAIL'}] Null block_geoid: {null_geoid}")

# 2. block_geoid length
bad_len = (block_level['block_geoid'].str.len() != 15).sum()
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid block_geoid length: {bad_len}")

# 3. State FIPS
expected_fips = {'06', '13', '17', '36', '48'}
actual_fips = set(block_level['state_fips'].unique())
fips_ok = actual_fips == expected_fips
print(f"[{'PASS' if fips_ok else 'FAIL'}] State FIPS values: {sorted(actual_fips)}")

# 4. Duplicates
dup_count = block_level.duplicated(subset=['block_geoid', 'provider_id', 'technology']).sum()
print(f"[{'PASS' if dup_count == 0 else 'WARN'}] Duplicate (block, provider, tech) combos: {dup_count:,}")

# 5. Null holding_company
null_company = block_level['holding_company'].isna().sum()
print(f"[{'PASS' if null_company == 0 else 'FAIL'}] Null holding_company: {null_company}")

# 6. Technology mapping
null_tech = block_level['technology_name'].isna().sum()
print(f"[{'PASS' if null_tech == 0 else 'FAIL'}] Null technology_name: {null_tech}")

# 7. Filing period
fp_ok = (block_level['filing_period'] == FILING_PERIOD).all()
print(f"[{'PASS' if fp_ok else 'FAIL'}] Filing period = {FILING_PERIOD}: {fp_ok}")

# 8. Speed ranges
print(f"\n--- Speed Ranges ---")
print(f"Download: {block_level['max_download'].min()} - {block_level['max_download'].max()}")
print(f"Upload:   {block_level['max_upload'].min()} - {block_level['max_upload'].max()}")

# 9. Service flags
print(f"\n--- Service Flags ---")
print(f"Serves residential: {block_level['serves_residential'].sum():,}")
print(f"Serves business:    {block_level['serves_business'].sum():,}")

=== VALIDATION REPORT ===

[PASS] Null block_geoid: 0
[PASS] Invalid block_geoid length: 0
[PASS] State FIPS values: ['06', '13', '17', '36', '48']
[WARN] Duplicate (block, provider, tech) combos: 17
[PASS] Null holding_company: 0
[PASS] Null technology_name: 0
[PASS] Filing period = 2022-06: True

--- Speed Ranges ---
Download: 0 - 500000
Upload:   0 - 500000

--- Service Flags ---
Serves residential: 748,080
Serves business:    737,536


## 10. Export

In [10]:
### Reorder columns to match 2025 schema + filing_period
block_level = block_level[[
    'block_geoid', 'state_fips', 'county_fips', 'tract_fips',
    'provider_id', 'holding_company', 'technology', 'technology_name',
    'max_download', 'max_upload', 'serves_residential', 'serves_business',
    'low_latency', 'filing_period'
]]

output_path = os.path.join(OUTPUT_DIR, 'all_states_block_level_availability_202206.csv')
block_level.to_csv(output_path, index=False)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Saved to: {output_path}")
print(f"Rows:     {block_level.shape[0]:,}")
print(f"Columns:  {block_level.shape[1]} — {list(block_level.columns)}")
print(f"Size:     {file_size_mb:.1f} MB")

### Quick verify
print(f"\n--- Verify (first 5 rows) ---")
verify = pd.read_csv(output_path, dtype={'block_geoid': str, 'state_fips': str,
                                          'county_fips': str, 'tract_fips': str,
                                          'provider_id': str}, nrows=5)
print(verify)

Saved to: ../../output/all_states_block_level_availability_202206.csv
Rows:     840,512
Columns:  14 — ['block_geoid', 'state_fips', 'county_fips', 'tract_fips', 'provider_id', 'holding_company', 'technology', 'technology_name', 'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency', 'filing_period']
Size:     103.1 MB

--- Verify (first 5 rows) ---
       block_geoid state_fips county_fips   tract_fips provider_id  \
0  060014001001010         06       06001  06001400100      130077   
1  060014001001010         06       06001  06001400100      190425   
2  060014001001011         06       06001  06001400100      130077   
3  060014001001013         06       06001  06001400100      130077   
4  060014001001013         06       06001  06001400100      190425   

      holding_company  technology               technology_name  max_download  \
0           AT&T Inc.          50  Fiber to the Premises (FTTP)          1000   
1  Sonic Telecom, LLC          50  